# Candidat de titre avant reranking

Ce test utilise exclusivement le corpus et les index stables. Pour chaque question, au plus une page dont le titre correspond fortement est ajoutée aux 15 passages évalués par le cross-encoder. Elle n'est jamais forcée dans le top 5.


In [1]:
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
import hashlib, json, logging, sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import load_config
from embeddings import EmbeddingModel
from evaluation import load_golden_set, RetrievalEvaluator
from retriever import HybridRetriever, Reranker
from vectorstore import VectorStoreManager

logging.basicConfig(level=logging.INFO)
cfg = load_config(ROOT / 'config.yaml')
test_cfg = cfg.model_copy(deep=True)
test_cfg.retrieval.title_candidate_enabled = True
test_cfg.retrieval.title_candidate_top_k = 1
test_cfg.retrieval.title_candidate_min_coverage = 0.60


In [2]:
def tree_fingerprint(root):
    digest = hashlib.sha256()
    for path in sorted(root.rglob('*')):
        if path.is_file():
            digest.update(str(path.relative_to(root)).encode('utf-8'))
            with open(path, 'rb') as handle:
                for block in iter(lambda: handle.read(1024 * 1024), b''):
                    digest.update(block)
    return digest.hexdigest()

real_lancedb = cfg.vectorstore.uri_path
real_bm25 = cfg.vectorstore.bm25_index_full_path
lancedb_hash_before = tree_fingerprint(real_lancedb)
bm25_hash_before = hashlib.sha256(real_bm25.read_bytes()).hexdigest()

embedding_model = EmbeddingModel(test_cfg.embeddings)
store = VectorStoreManager(test_cfg)
assert store.load_sparse(), 'Index BM25 stable introuvable.'
reranker = Reranker(test_cfg.reranker.model_name, test_cfg.reranker.device)
retriever = HybridRetriever(test_cfg, embedding_model, store, reranker)
pipeline = SimpleNamespace(retriever=retriever)
all_examples = load_golden_set(test_cfg.evaluation.expanded_golden_set_full_path)
_ = retriever.retrieve('Quelles formations propose l’ENSEA ?')
print(f'Index stable : {store.dense.count()} vecteurs, {len(store.sparse.chunk_lookup)} chunks BM25')
print(f'Entrées de titres uniques : {len(retriever._title_entries)}')


INFO:FlagEmbedding.finetune.embedder.encoder_only.m3.runner:loading existing colbert_linear and sparse_linear---------
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
INFO:vectorstore:Cache RAM des vecteurs chargé : 4459 vecteurs (17.4 Mo).
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
INFO:retriever:Retrieval 5.263s | query=0.039s dense=0.026s bm25=0.003s fusion=0.000s vectors+MMR=1.087s rerank=4.108s | 5 résultat(s)


Index stable : 4459 vecteurs, 4459 chunks BM25
Entrées de titres uniques : 1162


In [3]:
baselines = {
    'regression': {'mrr': 0.7183333333333334, 'hit_rate@5': 0.95, 'mean_latency_seconds': 0.3943393000001379},
    'development': {'mrr': 0.7875, 'hit_rate@5': 0.85, 'mean_latency_seconds': 0.3963170174996776},
}
benchmark_results = {}
benchmark_details = {}
comparisons = []
for split in ('regression', 'development'):
    examples = [example for example in all_examples if example.split == split]
    evaluator = RetrievalEvaluator(test_cfg.evaluation)
    summary = evaluator.evaluate(pipeline, examples)
    benchmark_results[split] = summary
    benchmark_details[split] = evaluator.last_details
    print(f'\n{split.upper()}')
    display(pd.Series(summary, name='valeur').to_frame())
    details = pd.DataFrame(evaluator.last_details)
    misses = details[details['should_answer'] & details['first_relevant_rank'].isna()]
    print(f'Échecs stricts : {len(misses)}/{len(details)}')
    display(misses[['id', 'question', 'category', 'top_title', 'top_url', 'relevant_urls']])
    for metric in ('mrr', 'hit_rate@5', 'mean_latency_seconds'):
        comparisons.append({
            'split': split, 'metric': metric,
            'référence': baselines[split][metric],
            'candidat_titre': summary[metric],
            'écart': summary[metric] - baselines[split][metric],
        })
display(pd.DataFrame(comparisons))


INFO:retriever:Retrieval 0.425s | query=0.038s dense=0.017s bm25=0.004s fusion=0.000s vectors+MMR=0.011s rerank=0.355s | 5 résultat(s)
INFO:retriever:Retrieval 0.421s | query=0.031s dense=0.018s bm25=0.004s fusion=0.000s vectors+MMR=0.010s rerank=0.359s | 5 résultat(s)
INFO:retriever:Retrieval 0.435s | query=0.043s dense=0.016s bm25=0.005s fusion=0.000s vectors+MMR=0.010s rerank=0.361s | 5 résultat(s)
INFO:retriever:Retrieval 0.377s | query=0.037s dense=0.016s bm25=0.003s fusion=0.000s vectors+MMR=0.010s rerank=0.312s | 5 résultat(s)
INFO:retriever:Retrieval 0.399s | query=0.034s dense=0.015s bm25=0.003s fusion=0.000s vectors+MMR=0.010s rerank=0.336s | 5 résultat(s)
INFO:retriever:Retrieval 0.462s | query=0.046s dense=0.017s bm25=0.005s fusion=0.000s vectors+MMR=0.009s rerank=0.384s | 5 résultat(s)
INFO:retriever:Retrieval 0.403s | query=0.042s dense=0.017s bm25=0.003s fusion=0.000s vectors+MMR=0.010s rerank=0.330s | 5 résultat(s)
INFO:retriever:Retrieval 0.430s | query=0.051s dense=0.


REGRESSION


,valeur
questions,20.000000
answerable_questions,20.000000
unanswerable_questions,0.000000
mrr,0.718333
mean_latency_seconds,0.410258
p50_latency_seconds,0.420624
p95_latency_seconds,0.434730
max_latency_seconds,0.461907
recall@3,0.666667
precision@3,0.325000


Échecs stricts : 1/20


,id,question,category,top_title,top_url,relevant_urls
8,reg-009,Quels sont les laboratoires de recherche assoc...,recherche,IRD : Institut de Recherche pour le Développem...,https://ensea.ed.ci/partenaires/institut-de-re...,[https://ensea.ed.ci/partenaires/laboratoire-d...


...


DEVELOPMENT


,valeur
questions,40.000000
answerable_questions,40.000000
unanswerable_questions,0.000000
mrr,0.812500
mean_latency_seconds,0.413490
p50_latency_seconds,0.427469
p95_latency_seconds,0.467915
max_latency_seconds,0.475293
recall@3,0.716667
precision@3,0.370833


Échecs stricts : 5/40


,id,question,category,top_title,top_url,relevant_urls
13,ext-014,Où consulter la liste des doctorants de l'ENSEA ?,doctorat,Admission Doctorat - ENSEA,https://ensea360.ensea.ed.ci/formation-continu...,[https://ensea.ed.ci/formation-doctorale/docto...
14,ext-015,L'ENSEA organise-t-elle des séminaires de rech...,recherche,765106d61b6f9473,https://ensea.ed.ci/wp-content/uploads/2024/06...,[https://ensea360.ensea.ed.ci/formation-contin...
15,ext-016,Quels organismes sont partenaires de recherche...,cooperation,Nos statistiques - ENSEA,https://ensea360.ensea.ed.ci/nos-statistiques/,[https://ensea.ed.ci/cooperation/partenaires-d...
26,ext-027,Où sont indiqués les coûts des formations init...,financement,,,[https://ensea.ed.ci/formation-initiale/couts/]
27,ext-028,Où puis-je consulter les tarifs de l'ENSEA ?,financement,Client entreprise - ENSEA,https://ensea360.ensea.ed.ci/formation-continu...,[https://ensea.ed.ci/faq-items/quels-sont-les-...


,split,metric,référence,candidat_titre,écart
0,regression,mrr,0.718333,0.718333,0.000000
1,regression,hit_rate@5,0.950000,0.950000,0.000000
2,regression,mean_latency_seconds,0.394339,0.410258,0.015919
3,development,mrr,0.787500,0.812500,0.025000
4,development,hit_rate@5,0.850000,0.875000,0.025000
5,development,mean_latency_seconds,0.396317,0.413490,0.017173


In [4]:
diagnostic_questions = [
    "Que propose le Master en Management de l'ENSEA ?",
    "Comment est organisée la formation doctorale de l'ENSEA ?",
    "Où consulter la liste des doctorants de l'ENSEA ?",
    "L'ENSEA organise-t-elle des séminaires de recherche ?",
]
diagnostics = []
for question in diagnostic_questions:
    results = retriever.retrieve(question)
    title_rows = [row for row in retriever.last_debug_rows if row.get('mmr_rank') and not row.get('rrf_rank')]
    diagnostics.append({
        'question': question,
        'candidat_titre_ajouté': retriever.last_timings.get('title_candidates_added'),
        'titre_candidat': title_rows[0]['title'] if title_rows else '',
        'rang_final': title_rows[0]['final_rank'] if title_rows else None,
        'top_final': results[0].title if results else '',
    })
display(pd.DataFrame(diagnostics))


INFO:retriever:Retrieval 0.384s | query=0.028s dense=0.015s bm25=0.005s fusion=0.000s vectors+MMR=0.011s rerank=0.325s | 5 résultat(s)
INFO:retriever:Retrieval 0.401s | query=0.053s dense=0.014s bm25=0.004s fusion=0.000s vectors+MMR=0.011s rerank=0.319s | 5 résultat(s)
INFO:retriever:Retrieval 0.428s | query=0.050s dense=0.014s bm25=0.003s fusion=0.000s vectors+MMR=0.010s rerank=0.352s | 3 résultat(s)
INFO:retriever:Retrieval 0.433s | query=0.030s dense=0.015s bm25=0.006s fusion=0.000s vectors+MMR=0.010s rerank=0.372s | 5 résultat(s)


,question,candidat_titre_ajouté,titre_candidat,rang_final,top_final
0,Que propose le Master en Management de l'ENSEA ?,1,,None,Master en Management - ENSEA
1,Comment est organisée la formation doctorale d...,0,,None,Formation doctorale - ENSEA
2,Où consulter la liste des doctorants de l'ENSEA ?,0,,None,Admission Doctorat - ENSEA
3,L'ENSEA organise-t-elle des séminaires de rech...,0,,None,765106d61b6f9473


In [5]:
quality_ok = all(
    benchmark_results[split]['mrr'] >= baselines[split]['mrr'] - 1e-9
    and benchmark_results[split]['hit_rate@5'] >= baselines[split]['hit_rate@5'] - 1e-9
    for split in baselines
)
latency_ok = all(
    benchmark_results[split]['mean_latency_seconds'] <= baselines[split]['mean_latency_seconds'] * 1.15
    for split in baselines
)
has_gain = any(
    benchmark_results[split]['mrr'] > baselines[split]['mrr'] + 1e-9
    or benchmark_results[split]['hit_rate@5'] > baselines[split]['hit_rate@5'] + 1e-9
    for split in baselines
)
promotion_recommended = quality_ok and latency_ok and has_gain
result_dir = ROOT / 'data' / 'experiments' / f"retriever_title_candidate_{datetime.now():%Y%m%d_%H%M%S}"
result_dir.mkdir(parents=True, exist_ok=False)
result_path = result_dir / 'benchmark_results.json'
result_path.write_text(json.dumps({
    'benchmarks': benchmark_results,
    'details': benchmark_details,
    'diagnostics': diagnostics,
    'promotion_recommended': promotion_recommended,
}, ensure_ascii=False, indent=2), encoding='utf-8')
assert tree_fingerprint(real_lancedb) == lancedb_hash_before
assert hashlib.sha256(real_bm25.read_bytes()).hexdigest() == bm25_hash_before
print({'quality_ok': quality_ok, 'latency_ok': latency_ok, 'has_gain': has_gain, 'promotion_recommended': promotion_recommended})
print('Corpus, LanceDB et BM25 réels strictement intacts.')
print(f'Résultats : {result_path}')


{'quality_ok': True, 'latency_ok': True, 'has_gain': True, 'promotion_recommended': True}
Corpus, LanceDB et BM25 réels strictement intacts.
Résultats : <PROJECT_ROOT>\data\experiments\retriever_title_candidate_20260803_132627\benchmark_results.json
